# Partie 2B V15-A — REBUILD complet — Planning_TL

Notebook reconstruit de zéro.

## Principe métier
- Entrée : `validation_domiciliations_V15_A.xlsx` corrigé/validé après Partie 2A.
- Le **CTS corrigé** est la référence métier pour le Planning TL.
- DOM / CTR / TTR ne réarbitrent plus les valeurs en Partie 2B.
- Le compte et le numéro de domiciliation viennent des valeurs finales de Partie 2A.
- Pour les augmentations, la période d'effet vient du fichier `parametres_augmentation.xlsx`.
- `TYPE_VERSION` distingue `NOUVEAU_CONTRAT` et `AUGMENTATION`.
- `NUMERO_AUGMENTATION` vaut 0 pour le nouveau contrat puis 1, 2, 3... pour les augmentations successives.
- Une augmentation remplace la version précédente à compter du premier jour du mois d'effet.
- ATF, autorisations et exécutions seront ajoutées ultérieurement.


In [ ]:
from pathlib import Path
from datetime import date, datetime, timedelta
import calendar
import hashlib
import math
import re
import pandas as pd
import numpy as np

VERSION = "V15A_PART2B_REBUILD_PLANNING_TL_V1"

ROOT = Path("/mnt/data/document_pipeline/V15_A")
INPUT_XLSX = ROOT / "02_postprocessing" / "validation_domiciliations_V15_A.xlsx"
PARAM_AUG_XLSX = Path("/mnt/data/parametres_augmentation.xlsx")

OUTPUT_DIR = ROOT / "03_planning_tl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_XLSX = OUTPUT_DIR / "Planning_TL_V15A_REBUILD.xlsx"
OUTPUT_CSV = OUTPUT_DIR / "Planning_TL_V15A_REBUILD.csv"
VERSIONS_CSV = OUTPUT_DIR / "Base_versions_CTS_V15A.csv"
REJETS_CSV = OUTPUT_DIR / "Rejets_Planning_TL_V15A.csv"

print("Version :", VERSION)
print("Entrée  :", INPUT_XLSX)
print("Param.  :", PARAM_AUG_XLSX)
print("Sortie  :", OUTPUT_XLSX)


In [ ]:
def is_blank(v):
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    return str(v).strip().lower() in ("", "nan", "none", "null")

def clean(v):
    return None if is_blank(v) else v

def parse_date(v):
    if is_blank(v):
        return None
    if isinstance(v, pd.Timestamp):
        return v.date()
    if isinstance(v, datetime):
        return v.date()
    if isinstance(v, date):
        return v

    s = str(v).strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%d.%m.%Y"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            pass

    x = pd.to_datetime(v, dayfirst=True, errors="coerce")
    return None if pd.isna(x) else x.date()

def parse_amount(v):
    if is_blank(v):
        return None
    if isinstance(v, (int, float, np.number)):
        return float(v)

    s = str(v).upper().strip()
    s = s.replace("\\u00a0", "").replace(" ", "")
    s = s.replace("DZD", "").replace("DA", "")
    s = s.replace(":", ".")

    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        parts = s.split(",")
        if len(parts[-1]) <= 2:
            s = "".join(parts[:-1]) + "." + parts[-1]
        else:
            s = "".join(parts)
    elif s.count(".") > 1:
        parts = s.split(".")
        if len(parts[-1]) <= 2:
            s = "".join(parts[:-1]) + "." + parts[-1]
        else:
            s = "".join(parts)

    s = re.sub(r"[^0-9.+-]", "", s)
    try:
        return float(s)
    except Exception:
        return None

def normalize_decision(v):
    if is_blank(v):
        return None
    s = str(v).strip().upper()
    s = s.replace("É","E")
    mapping = {
        "VALIDE":"VALIDEE", "VALIDEE":"VALIDEE",
        "CORRIGE":"CORRIGEE", "CORRIGEE":"CORRIGEE",
        "REJETE":"REJETEE", "REJETEE":"REJETEE"
    }
    return mapping.get(s, s)

def first_value(row, candidates):
    for c in candidates:
        if c in row.index and not is_blank(row.get(c)):
            return row.get(c)
    return None


In [ ]:
# 1. Chargement Partie 2A corrigée
if not INPUT_XLSX.exists():
    raise FileNotFoundError(f"Fichier Partie 2A absent : {INPUT_XLSX}")

xls = pd.ExcelFile(INPUT_XLSX)
print("Feuilles Partie 2A :", xls.sheet_names)

if "DOSSIERS_A_VALIDER" not in xls.sheet_names:
    raise RuntimeError("Feuille DOSSIERS_A_VALIDER absente")
if "A_VALIDER_MANUELLEMENT" not in xls.sheet_names:
    raise RuntimeError("Feuille A_VALIDER_MANUELLEMENT absente")

DOSSIERS = pd.read_excel(INPUT_XLSX, sheet_name="DOSSIERS_A_VALIDER", dtype=object)
MANUEL = pd.read_excel(INPUT_XLSX, sheet_name="A_VALIDER_MANUELLEMENT", dtype=object)

if "FICHIER" not in DOSSIERS.columns:
    raise RuntimeError("Colonne FICHIER absente de DOSSIERS_A_VALIDER")

print("Dossiers :", len(DOSSIERS))
print("Corrections/validations :", len(MANUEL))


In [ ]:
# 2. Application des corrections manuelles
BASE = DOSSIERS.copy()
audit = []
dossiers_rejetes = set()

for _, r in MANUEL.iterrows():
    fichier = clean(r.get("FICHIER"))
    champ = clean(r.get("CHAMP"))
    decision = normalize_decision(r.get("DECISION_MANUELLE"))

    if fichier is None or champ is None or decision is None:
        continue

    mask = BASE["FICHIER"].astype(str) == str(fichier)
    if not mask.any():
        audit.append({
            "FICHIER":fichier, "CHAMP":champ, "DECISION":decision,
            "STATUT_APPLICATION":"FICHIER_NON_TROUVE"
        })
        continue

    if decision == "REJETEE":
        dossiers_rejetes.add(str(fichier))
        audit.append({
            "FICHIER":fichier, "CHAMP":champ, "DECISION":decision,
            "STATUT_APPLICATION":"DOSSIER_REJETE"
        })
        continue

    valeur_validee = clean(r.get("VALEUR_VALIDEE"))
    valeur_proposee = clean(r.get("VALEUR_PROPOSEE"))
    valeur_normalisee = clean(r.get("NORMALIZED"))
    valeur_raw = clean(r.get("RAW"))

    if decision == "CORRIGEE":
        valeur = valeur_validee
        if valeur is None:
            audit.append({
                "FICHIER":fichier, "CHAMP":champ, "DECISION":decision,
                "STATUT_APPLICATION":"CORRECTION_SANS_VALEUR_VALIDEE"
            })
            continue
    else:
        valeur = (
            valeur_validee if valeur_validee is not None else
            valeur_proposee if valeur_proposee is not None else
            valeur_normalisee if valeur_normalisee is not None else
            valeur_raw
        )

    if champ not in BASE.columns:
        BASE[champ] = None

    ancienne = BASE.loc[mask, champ].iloc[0]
    if valeur is not None:
        BASE.loc[mask, champ] = valeur

    audit.append({
        "FICHIER":fichier,
        "CHAMP":champ,
        "DECISION":decision,
        "ANCIENNE_VALEUR":ancienne,
        "VALEUR_FINALE":BASE.loc[mask, champ].iloc[0],
        "STATUT_APPLICATION":"APPLIQUEE"
    })

AUDIT = pd.DataFrame(audit)
print("Décisions traitées :", len(AUDIT))
print("Dossiers rejetés :", len(dossiers_rejetes))


In [ ]:
# 3. Calendrier officiel des mois d'augmentation
if not PARAM_AUG_XLSX.exists():
    raise FileNotFoundError(f"Fichier paramètre augmentation absent : {PARAM_AUG_XLSX}")

PARAM_AUG = pd.read_excel(PARAM_AUG_XLSX, dtype=object)
colmap = {str(c).strip().upper(): c for c in PARAM_AUG.columns}

required = {"ANNEE_AUGMENTATION", "MOIS_AUGMENTATION"}
if not required.issubset(colmap):
    raise RuntimeError(
        "parametres_augmentation.xlsx doit contenir "
        "ANNEE_AUGMENTATION et MOIS_AUGMENTATION"
    )

AUG_MONTH_BY_YEAR = {}
for _, r in PARAM_AUG.iterrows():
    yv = r[colmap["ANNEE_AUGMENTATION"]]
    mv = r[colmap["MOIS_AUGMENTATION"]]
    if is_blank(yv) or is_blank(mv):
        continue
    y = int(float(yv))
    m = int(float(mv))
    if not 1 <= m <= 12:
        raise ValueError(f"Mois d'augmentation invalide : année={y}, mois={m}")
    AUG_MONTH_BY_YEAR[y] = m

print("Calendrier augmentation :", AUG_MONTH_BY_YEAR)


In [ ]:
# 4. Constitution des versions CTS
versions = []
rejets = []

for _, r in BASE.iterrows():
    fichier = str(r.get("FICHIER"))

    if fichier in dossiers_rejetes:
        rejets.append({"FICHIER":fichier, "MOTIF":"REJETE_EN_PARTIE_2A"})
        continue

    type_dossier = str(r.get("TYPE_DOSSIER") or "").strip().upper()
    if type_dossier not in ("NOUVEAU_CONTRAT", "AUGMENTATION"):
        rejets.append({"FICHIER":fichier, "MOTIF":"TYPE_DOSSIER_NON_RESOLU"})
        continue

    # Référence métier : CTS corrigé
    debut_permis = parse_date(r.get("CTS_DATE_DEBUT_VALIDITE_PERMIS"))
    fin_permis = parse_date(r.get("CTS_DATE_FIN_VALIDITE_PERMIS"))
    salaire_ancien = parse_amount(r.get("CTS_SALAIRE_NET_ANCIEN"))
    salaire_nouveau = parse_amount(r.get("CTS_SALAIRE_NET"))
    part_transferable = parse_amount(r.get("CTS_PART_TRANSFERABLE"))

    # Compte + domiciliation : valeurs finales Partie 2A
    compte = first_value(r, [
        "DOM_COMPTE_LOCAL_NORMALIZED",
        "DOM_COMPTE_LOCAL"
    ])
    numero_dom = first_value(r, [
        "NUMERO_DOMICILIATION_VALIDE",
        "NUMERO_DOMICILIATION_PROPOSE",
        "NUMERO_DOMICILIATION_NORMALIZED"
    ])

    motifs = []
    if debut_permis is None:
        motifs.append("CTS_DATE_DEBUT_VALIDITE_PERMIS_ABSENTE")
    if fin_permis is None:
        motifs.append("CTS_DATE_FIN_VALIDITE_PERMIS_ABSENTE")
    if debut_permis and fin_permis and fin_permis < debut_permis:
        motifs.append("FIN_PERMIS_AVANT_DEBUT")
    if salaire_nouveau is None:
        motifs.append("CTS_SALAIRE_NET_ABSENT")
    if part_transferable is None:
        motifs.append("CTS_PART_TRANSFERABLE_ABSENTE")
    if is_blank(compte):
        motifs.append("COMPTE_FINAL_ABSENT")
    if is_blank(numero_dom):
        motifs.append("NUMERO_DOMICILIATION_FINAL_ABSENT")

    annee_aug = None
    mois_aug = None

    if type_dossier == "NOUVEAU_CONTRAT":
        date_effet = debut_permis
    else:
        # L'année résolue en Partie 2A prime.
        if not is_blank(r.get("AUGMENTATION_YEAR")):
            try:
                annee_aug = int(float(r.get("AUGMENTATION_YEAR")))
            except Exception:
                annee_aug = None

        # Fallback seulement pour identifier l'année.
        if annee_aug is None:
            ddoc = parse_date(r.get("CTS_DATE_DOCUMENT"))
            if ddoc:
                annee_aug = ddoc.year

        if annee_aug is not None:
            mois_aug = AUG_MONTH_BY_YEAR.get(annee_aug)

        if annee_aug is None or mois_aug is None:
            date_effet = None
            motifs.append("PERIODE_AUGMENTATION_PARAMETREE_ABSENTE")
        else:
            date_effet = date(annee_aug, mois_aug, 1)

    if motifs:
        rejets.append({"FICHIER":fichier, "MOTIF":" | ".join(motifs)})
        continue

    # Identifiant de domiciliation pour regrouper nouveau contrat + augmentations.
    dom_key = f"{str(numero_dom).strip()}|{str(compte).strip()}"
    dom_id = hashlib.sha256(dom_key.encode("utf-8")).hexdigest()[:20]

    versions.append({
        "DOM_ID":dom_id,
        "FICHIER":fichier,
        "NUMERO_DOMICILIATION":numero_dom,
        "DOM_COMPTE_LOCAL":compte,
        "SIEGE_RACINE":r.get("SIEGE_RACINE"),

        "TYPE_VERSION":type_dossier,
        "DATE_EFFET_VERSION":date_effet,
        "ANNEE_AUGMENTATION":annee_aug,
        "MOIS_AUGMENTATION_PARAMETRE":mois_aug,

        "CTS_REFERENCE_DOCUMENT":r.get("CTS_REFERENCE_DOCUMENT"),
        "CTS_NOM_PRENOM_TRAVAILLEUR":r.get("CTS_NOM_PRENOM_TRAVAILLEUR"),
        "CTS_NUMERO_PERMIS_TRAVAIL":r.get("CTS_NUMERO_PERMIS_TRAVAIL"),
        "CTS_DATE_DEBUT_VALIDITE_PERMIS":debut_permis,
        "CTS_DATE_FIN_VALIDITE_PERMIS":fin_permis,

        "CTS_SALAIRE_NET_ANCIEN":salaire_ancien,
        "CTS_SALAIRE_NET":salaire_nouveau,
        "CTS_PART_TRANSFERABLE":part_transferable,
        "CTS_PART_PAYABLE_DZD":parse_amount(r.get("CTS_PART_PAYABLE_DZD")),
        "CTS_DATE_DOCUMENT":parse_date(r.get("CTS_DATE_DOCUMENT")),

        "DQ_SCORE":r.get("DQ_SCORE"),
        "DOSSIER_STATUS":r.get("DOSSIER_STATUS")
    })

VERSIONS = pd.DataFrame(versions)
REJETS = pd.DataFrame(rejets)

print("Versions CTS valides :", len(VERSIONS))
print("Rejets :", len(REJETS))


In [ ]:
# 5. Numérotation des augmentations
if not VERSIONS.empty:
    VERSIONS["_ORDRE_TYPE"] = VERSIONS["TYPE_VERSION"].map({
        "NOUVEAU_CONTRAT": 0,
        "AUGMENTATION": 1
    })

    VERSIONS = VERSIONS.sort_values(
        ["DOM_ID", "DATE_EFFET_VERSION", "_ORDRE_TYPE", "FICHIER"]
    ).reset_index(drop=True)

    VERSIONS["NUMERO_AUGMENTATION"] = 0
    VERSIONS["NOMBRE_TOTAL_AUGMENTATIONS"] = 0

    for dom_id, indexes in VERSIONS.groupby("DOM_ID").groups.items():
        indexes = list(indexes)
        n = 0
        for idx in indexes:
            if VERSIONS.at[idx, "TYPE_VERSION"] == "AUGMENTATION":
                n += 1
                VERSIONS.at[idx, "NUMERO_AUGMENTATION"] = n
        for idx in indexes:
            VERSIONS.at[idx, "NOMBRE_TOTAL_AUGMENTATIONS"] = n

    VERSIONS.drop(columns=["_ORDRE_TYPE"], inplace=True)

VERSIONS.head()


In [ ]:
# 6. Fenêtre d'application de chaque version
# Une augmentation coupe la version précédente la veille de sa date d'effet.
windows = []
window_rejects = []

for dom_id, group in VERSIONS.groupby("DOM_ID", sort=False):
    group = group.sort_values(
        ["DATE_EFFET_VERSION", "NUMERO_AUGMENTATION", "FICHIER"]
    ).reset_index(drop=True)

    for i, row in group.iterrows():
        start = max(
            row["DATE_EFFET_VERSION"],
            row["CTS_DATE_DEBUT_VALIDITE_PERMIS"]
        )
        end = row["CTS_DATE_FIN_VALIDITE_PERMIS"]

        if i + 1 < len(group):
            next_effect = group.loc[i + 1, "DATE_EFFET_VERSION"]
            end = min(end, next_effect - timedelta(days=1))

        if end < start:
            window_rejects.append({
                "FICHIER":row["FICHIER"],
                "MOTIF":"FENETRE_VERSION_VIDE_APRES_DECOUPAGE"
            })
            continue

        d = row.to_dict()
        d["DATE_DEBUT_APPLICABLE"] = start
        d["DATE_FIN_APPLICABLE"] = end
        d["VERSION_ID"] = hashlib.sha256(
            f'{dom_id}|{row["FICHIER"]}|{row["TYPE_VERSION"]}|'
            f'{row["NUMERO_AUGMENTATION"]}|{start.isoformat()}'.encode("utf-8")
        ).hexdigest()[:24]
        windows.append(d)

VERSIONS_APPLICABLES = pd.DataFrame(windows)

if window_rejects:
    REJETS = pd.concat(
        [REJETS, pd.DataFrame(window_rejects)],
        ignore_index=True
    )

print("Versions applicables :", len(VERSIONS_APPLICABLES))


In [ ]:
# 7. Génération Planning_TL
def generate_periods(row):
    start = row["DATE_DEBUT_APPLICABLE"]
    end = row["DATE_FIN_APPLICABLE"]
    cursor = date(start.year, start.month, 1)
    rows = []

    while cursor <= end:
        nb_jours_mois = calendar.monthrange(cursor.year, cursor.month)[1]
        fin_mois = date(cursor.year, cursor.month, nb_jours_mois)

        seg_start = max(start, cursor)
        seg_end = min(end, fin_mois)

        if seg_start <= seg_end:
            mois_complet = seg_start == cursor and seg_end == fin_mois

            if mois_complet:
                partie = None
                periode = f"{cursor.year:04d}-{cursor.month:02d}"
            elif seg_start.day == 1:
                partie = "P1"
                periode = f"{cursor.year:04d}-{cursor.month:02d}P1"
            else:
                partie = "P2"
                periode = f"{cursor.year:04d}-{cursor.month:02d}P2"

            nb_jours = (seg_end - seg_start).days + 1
            coefficient = nb_jours / nb_jours_mois
            plafond = row["CTS_PART_TRANSFERABLE"]

            period_id = hashlib.sha256(
                f'{row["VERSION_ID"]}|{periode}'.encode("utf-8")
            ).hexdigest()[:24]

            rows.append({
                "PERIODE_ID":period_id,
                "DOM_ID":row["DOM_ID"],
                "VERSION_ID":row["VERSION_ID"],

                "FICHIER_SOURCE":row["FICHIER"],
                "NUMERO_DOMICILIATION":row["NUMERO_DOMICILIATION"],
                "DOM_COMPTE_LOCAL":row["DOM_COMPTE_LOCAL"],
                "SIEGE_RACINE":row.get("SIEGE_RACINE"),

                "NOM_TRAVAILLEUR":row.get("CTS_NOM_PRENOM_TRAVAILLEUR"),
                "NUMERO_PERMIS_TRAVAIL":row.get("CTS_NUMERO_PERMIS_TRAVAIL"),
                "CTS_REFERENCE_DOCUMENT":row.get("CTS_REFERENCE_DOCUMENT"),

                # Nouveau contrat / augmentation
                "TYPE_VERSION":row["TYPE_VERSION"],
                "NUMERO_AUGMENTATION":int(row["NUMERO_AUGMENTATION"]),
                "NOMBRE_TOTAL_AUGMENTATIONS":int(row["NOMBRE_TOTAL_AUGMENTATIONS"]),
                "ANNEE_AUGMENTATION":row.get("ANNEE_AUGMENTATION"),
                "MOIS_AUGMENTATION_PARAMETRE":row.get("MOIS_AUGMENTATION_PARAMETRE"),
                "DATE_EFFET_VERSION":row["DATE_EFFET_VERSION"].isoformat(),

                # Référence CTS
                "DATE_DEBUT_VALIDITE_PERMIS":row["CTS_DATE_DEBUT_VALIDITE_PERMIS"].isoformat(),
                "DATE_FIN_VALIDITE_PERMIS":row["CTS_DATE_FIN_VALIDITE_PERMIS"].isoformat(),

                # Fenêtre de cette version
                "DATE_DEBUT_APPLICABLE":start.isoformat(),
                "DATE_FIN_APPLICABLE":end.isoformat(),

                # Période TL
                "PERIODE_TL":periode,
                "MOIS_BASE":f"{cursor.year:04d}-{cursor.month:02d}",
                "PARTIE":partie,
                "DATE_DEBUT_SEGMENT":seg_start.isoformat(),
                "DATE_FIN_SEGMENT":seg_end.isoformat(),
                "NB_JOURS_SEGMENT":nb_jours,
                "NB_JOURS_MOIS":nb_jours_mois,
                "COEFFICIENT_PRORATA":round(coefficient, 8),

                # Données financières CTS
                "SALAIRE_NET_ANCIEN_CTS":row.get("CTS_SALAIRE_NET_ANCIEN"),
                "SALAIRE_NET_NOUVEAU_CTS":row.get("CTS_SALAIRE_NET"),
                "SALAIRE_NET_APPLICABLE":row.get("CTS_SALAIRE_NET"),
                "PART_TRANSFERABLE_CTS":plafond,
                "MONTANT_MAX_THEORIQUE":round(plafond * coefficient, 2),

                # Enrichissements futurs
                "ATF_REFERENCE":None,
                "AUTORISATION_REFERENCE":None,
                "MONTANT_AUTORISE_SAISI":None,
                "MONTANT_TRANSFERE":None,
                "SOLDE_RESTANT":None,
                "DATE_EXECUTION_TL":None,
                "STATUT_PERIODE":"BASE_THEORIQUE_CTS",
                "MOTIF_REJET":None
            })

        if cursor.month == 12:
            cursor = date(cursor.year + 1, 1, 1)
        else:
            cursor = date(cursor.year, cursor.month + 1, 1)

    return rows

planning_rows = []
for _, r in VERSIONS_APPLICABLES.iterrows():
    planning_rows.extend(generate_periods(r.to_dict()))

PLANNING_TL = pd.DataFrame(planning_rows)

if not PLANNING_TL.empty:
    PLANNING_TL = PLANNING_TL.sort_values(
        ["DOM_ID", "DATE_DEBUT_SEGMENT", "NUMERO_AUGMENTATION"]
    ).reset_index(drop=True)

print("Nombre de périodes :", len(PLANNING_TL))


In [ ]:
# 8. Contrôles Data Quality du Planning
dq_errors = []

if not PLANNING_TL.empty:
    if PLANNING_TL["PERIODE_ID"].duplicated().any():
        dq_errors.append("PERIODE_ID_DUPLIQUE")

    bad_coef = (
        (PLANNING_TL["COEFFICIENT_PRORATA"] <= 0) |
        (PLANNING_TL["COEFFICIENT_PRORATA"] > 1)
    )
    if bad_coef.any():
        dq_errors.append("COEFFICIENT_PRORATA_INVALIDE")

    # Une même version ne doit pas avoir deux fois la même période.
    dup = PLANNING_TL.duplicated(
        ["VERSION_ID", "PERIODE_TL"],
        keep=False
    )
    if dup.any():
        dq_errors.append("VERSION_PERIODE_DUPLIQUEE")

print("Contrôles Planning :", "OK" if not dq_errors else dq_errors)


In [ ]:
# 9. Exports
PLANNING_TL.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
VERSIONS_APPLICABLES.to_csv(VERSIONS_CSV, index=False, encoding="utf-8-sig")
REJETS.to_csv(REJETS_CSV, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    PLANNING_TL.to_excel(writer, sheet_name="Planning_TL", index=False)
    VERSIONS_APPLICABLES.to_excel(writer, sheet_name="BASE_VERSIONS_CTS", index=False)
    AUDIT.to_excel(writer, sheet_name="CORRECTIONS_APPLIQUEES", index=False)
    PARAM_AUG.to_excel(writer, sheet_name="PARAM_AUGMENTATION", index=False)
    REJETS.to_excel(writer, sheet_name="REJETS_PLANNING", index=False)

    regles = pd.DataFrame([
        ["VERSION", VERSION],
        ["SOURCE", "validation_domiciliations_V15_A.xlsx corrigé"],
        ["REFERENCE_METIER", "CTS corrigé/validé Partie 2A"],
        ["COMPTE", "Valeur finale Partie 2A"],
        ["NUMERO_DOMICILIATION", "Valeur finale/validée Partie 2A"],
        ["DATES_PERMIS", "CTS_DATE_DEBUT_VALIDITE_PERMIS / CTS_DATE_FIN_VALIDITE_PERMIS"],
        ["SALAIRE_ANCIEN", "CTS_SALAIRE_NET_ANCIEN"],
        ["SALAIRE_NOUVEAU", "CTS_SALAIRE_NET"],
        ["PART_TRANSFERABLE", "CTS_PART_TRANSFERABLE"],
        ["TYPE_VERSION", "NOUVEAU_CONTRAT / AUGMENTATION"],
        ["NUMERO_AUGMENTATION", "0 nouveau contrat; 1,2,3... augmentations"],
        ["NOMBRE_TOTAL_AUGMENTATIONS", "Nombre total d'augmentations de la domiciliation"],
        ["PERIODE_AUGMENTATION", "ANNEE_AUGMENTATION + MOIS_AUGMENTATION du fichier paramètre"],
        ["DATE_EFFET_AUGMENTATION", "Premier jour du mois paramétré"],
        ["REGLE_REMPLACEMENT", "Nouvelle version coupe la précédente la veille de sa date d'effet"],
        ["PRORATA", "NB_JOURS_SEGMENT / NB_JOURS_MOIS"],
        ["ATF_AUTORISATIONS", "Hors périmètre de cette version"]
    ], columns=["PARAMETRE", "REGLE"])
    regles.to_excel(writer, sheet_name="REGLES", index=False)

print("Terminé.")
print("Excel :", OUTPUT_XLSX)
print("CSV Planning :", OUTPUT_CSV)
print("CSV versions :", VERSIONS_CSV)
print("CSV rejets :", REJETS_CSV)
